In [7]:
# =============================================================
# Fase 5 — Dashboard interactivo y cartografía final
# Pipeline multilenguaje CGSM (2018–2025)
# Lina María Quintero Fonseca | Maestría en Geomática, UNAL
# =============================================================
import geemap
import ee
from ipyleaflet import Marker, DivIcon
import os

ee.Initialize()

aoi_coords = [
    [-74.8700, 11.0600], [-74.8200, 11.0700], [-74.7500, 11.0650],
    [-74.6800, 11.0600], [-74.6000, 11.0500], [-74.5300, 11.0400],
    [-74.4800, 11.0350], [-74.4300, 11.0300], [-74.3800, 11.0200],
    [-74.3200, 11.0100], [-74.2500, 10.9900], [-74.2000, 10.9500],
    [-74.1700, 10.9000], [-74.1500, 10.8500], [-74.1300, 10.8000],
    [-74.1200, 10.7500], [-74.1100, 10.7000], [-74.1000, 10.6500],
    [-74.1200, 10.6000], [-74.1500, 10.5500], [-74.2000, 10.5000],
    [-74.2500, 10.4500], [-74.3000, 10.4000], [-74.3500, 10.3500],
    [-74.4500, 10.3400], [-74.5500, 10.3800], [-74.6200, 10.4200],
    [-74.7000, 10.4800], [-74.7500, 10.5500], [-74.8000, 10.6500],
    [-74.8500, 10.7500], [-74.8700, 10.8500], [-74.8800, 10.9500],
    [-74.8700, 11.0600]
]

stations = {
    'Isla Boquerón': (-74.298, 10.962, 'INVEMAR'),
    'Punta Cerro': (-74.283, 10.973, 'INVEMAR'),
    'Punta Chino': (-74.305, 10.912, 'INVEMAR'),
    'Río Sevilla': (-74.325, 10.880, 'INVEMAR'),
    'Caño Palos': (-74.471, 10.758, 'INVEMAR'),
    'CP Pajarales': (-74.75, 10.85, 'Complementaria'),
    'Caño Clarín': (-74.55, 10.55, 'Complementaria'),
    'VIPIS': (-74.65, 11.02, 'Complementaria'),
}

def mask_s2(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    cmri = ndvi.subtract(ndwi).rename('CMRI')
    return image.addBands([ndvi, ndwi, cmri])

aoi = ee.Geometry.Polygon(aoi_coords)

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .map(mask_s2)
      .map(add_indices))

os.makedirs('../outputs/maps', exist_ok=True)

# --- Mapa ---
m = geemap.Map(
    center=[10.72, -74.45], zoom=10,
    draw_control=False, measure_control=False,
    fullscreen_control=True, scale_control=True,
    layout={'height': '700px'}
)

m.add_basemap('Esri.WorldImagery')
m.add_basemap('Esri.WorldTopoMap', shown=False)
m.add_basemap('OpenStreetMap', shown=False)

# NDVI por período
palette_ndvi = ['#8B0000', '#D32F2F', '#FF6F00', '#FDD835', '#7CB342', '#2E7D32', '#1B5E20']
vis_ndvi = {'min': -0.2, 'max': 0.8, 'palette': palette_ndvi}

ndvi_deg = s2.filterDate('2020-07-01', '2020-12-31').select('NDVI').median().clip(aoi)
ndvi_rec = s2.filterDate('2022-01-01', '2022-06-30').select('NDVI').median().clip(aoi)
ndvi_act = s2.filterDate('2024-07-01', '2025-06-30').select('NDVI').median().clip(aoi)
ndvi_change = ndvi_act.subtract(ndvi_deg)

vis_change = {'min': -0.4, 'max': 0.4, 'palette': ['#d73027','#f46d43','#fdae61','#ffffbf','#a6d96a','#66bd63','#1a9850']}

m.addLayer(ndvi_deg, vis_ndvi, 'NDVI Degradación (2020)', shown=False)
m.addLayer(ndvi_rec, vis_ndvi, 'NDVI Recuperación (2022)', shown=False)
m.addLayer(ndvi_act, vis_ndvi, 'NDVI Actual (2024-2025)', shown=True)
m.addLayer(ndvi_change, vis_change, 'Cambio NDVI (Actual - Degradación)', shown=False)

# Clasificación manglar por período
def manglar_mask(start, end):
    comp = s2.filterDate(start, end).median().clip(aoi)
    ndvi = comp.normalizedDifference(['B8', 'B4'])
    ndwi = comp.normalizedDifference(['B3', 'B8'])
    cmri = ndvi.subtract(ndwi)
    return cmri.gt(0.3).And(ndvi.gt(0.4)).selfMask()

m.addLayer(manglar_mask('2020-07-01','2020-12-31'), {'palette':['#D32F2F']}, 'Manglar Degradación', shown=False, opacity=0.7)
m.addLayer(manglar_mask('2022-01-01','2022-06-30'), {'palette':['#FF9800']}, 'Manglar Recuperación', shown=False, opacity=0.7)
m.addLayer(manglar_mask('2024-07-01','2025-06-30'), {'palette':['#1B5E20']}, 'Manglar Actual', shown=False, opacity=0.7)

# Inundación SAR
s1_dry = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi).filterDate('2020-01-01','2020-03-31')
    .filter(ee.Filter.eq('instrumentMode','IW'))
    .select('VH').median().clip(aoi))
s1_flood = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi).filterDate('2020-09-01','2020-10-31')
    .filter(ee.Filter.eq('instrumentMode','IW'))
    .select('VH').median().clip(aoi))
sar_diff = s1_dry.subtract(s1_flood)

m.addLayer(sar_diff.gt(3).selfMask(), {'palette':['#00BCD4']}, 'Inundación agua abierta', shown=False, opacity=0.7)
m.addLayer(sar_diff.lt(-2).selfMask(), {'palette':['#E040FB']}, 'Inundación bajo dosel', shown=False, opacity=0.7)

# AOI y estaciones
styled_aoi = ee.FeatureCollection([ee.Feature(aoi)]).style(color='FF3333', fillColor='00000000', width=2)
m.addLayer(styled_aoi, {}, 'Área de estudio')

invemar_pts = []
comple_pts = []
for name, (lon, lat, tipo) in stations.items():
    pt = ee.Feature(ee.Geometry.Point([lon, lat]).buffer(500), {'name': name})
    if tipo == 'INVEMAR':
        invemar_pts.append(pt)
    else:
        comple_pts.append(pt)

m.addLayer(ee.FeatureCollection(invemar_pts).style(color='FF3333', fillColor='FF333399', width=2), {}, 'Estaciones INVEMAR')
m.addLayer(ee.FeatureCollection(comple_pts).style(color='FFB300', fillColor='FFB30099', width=2), {}, 'Estaciones complementarias')

# Labels estaciones
for name, (lon, lat, tipo) in stations.items():
    color = '#FF4444' if tipo == 'INVEMAR' else '#FFB300'
    icon = DivIcon(
        html=f'<div style="font-size:10px; font-weight:bold; color:{color}; text-shadow:1px 1px 3px black, -1px -1px 3px black; white-space:nowrap; background:none !important; border:none !important; box-shadow:none !important;">{name}</div>',
        icon_size=[0,0], icon_anchor=[-10,25], class_name='')
    m.add(Marker(location=[lat,lon], icon=icon, draggable=False))

# Labels geográficos
geo_labels = {
    'Ciénaga Grande de Santa Marta': (10.84, -74.45, '#7EC8E3', 13),
    'Mar Caribe': (11.06, -74.50, '#A8D8EA', 15),
    'Complejo de Pajarales': (10.80, -74.74, '#A8E6A1', 10),
    'Isla de Salamanca': (11.035, -74.65, '#A8E6A1', 10),
}
for label, (lat, lon, color, size) in geo_labels.items():
    icon = DivIcon(
        html=f'<div style="font-size:{size}px; font-style:italic; color:{color}; text-shadow:1px 1px 3px black, -1px -1px 3px black; white-space:nowrap; background:none !important; border:none !important; box-shadow:none !important;">{label}</div>',
        icon_size=[0,0], icon_anchor=[60,10], class_name='')
    m.add(Marker(location=[lat,lon], icon=icon, draggable=False))

# Barra de color NDVI
m.add_colorbar(vis_ndvi, label='NDVI', position='bottomright')

print("Dashboard listo")
m

Dashboard listo


Map(center=[10.72, -74.45], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [9]:
m.to_html('../outputs/maps/dashboard_CGSM_final.html', title='Dashboard CGSM - Monitoreo de Manglar 2018-2025')
print("Dashboard exportado")
print("Tamaño:", round(os.path.getsize('../outputs/maps/dashboard_CGSM_final.html') / 1e6, 1), "MB")

Dashboard exportado
Tamaño: 2.0 MB


In [8]:
# --- Agregar capa ganancia/pérdida al dashboard existente ---

# Restricciones espaciales
srtm = ee.Image('USGS/SRTMGL1_003').clip(aoi)
elev_mask = srtm.lt(10)
jrc_water = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').clip(aoi)
water = jrc_water.gt(30)
water_dist = water.fastDistanceTransform().sqrt().multiply(30)
near_water = water_dist.lt(3000)

# Clasificación con umbrales optimizados
def manglar_optimized(start, end):
    comp = s2.filterDate(start, end).median().clip(aoi)
    ndvi = comp.normalizedDifference(['B8', 'B4'])
    return ndvi.gt(0.70).And(elev_mask).And(near_water).selfMask()

manglar_deg = manglar_optimized('2020-07-01', '2020-12-31')
manglar_act = manglar_optimized('2024-07-01', '2025-06-30')

deg_binary = manglar_deg.unmask(0).gt(0)
act_binary = manglar_act.unmask(0).gt(0)

perdida = deg_binary.And(act_binary.Not()).selfMask()
estable = deg_binary.And(act_binary).selfMask()
ganancia = deg_binary.Not().And(act_binary).selfMask()

# Calcular áreas
print("=== GANANCIA/PÉRDIDA DE MANGLAR ===\n")

area_per = perdida.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
).getInfo()
area_est = estable.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
).getInfo()
area_gan = ganancia.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e13
).getInfo()

per_km2 = list(area_per.values())[0] / 1e6
est_km2 = list(area_est.values())[0] / 1e6
gan_km2 = list(area_gan.values())[0] / 1e6
neto_km2 = gan_km2 - per_km2

print(f"  Pérdida:   {per_km2:.1f} km²")
print(f"  Estable:   {est_km2:.1f} km²")
print(f"  Ganancia:  {gan_km2:.1f} km²")
print(f"  Cambio neto: {'+' if neto_km2 > 0 else ''}{neto_km2:.1f} km²")

# Agregar al mapa existente
m.addLayer(perdida, {'palette': ['#D32F2F']}, 'Pérdida manglar', shown=False, opacity=0.8)
m.addLayer(estable, {'palette': ['#2E7D32']}, 'Manglar estable', shown=False, opacity=0.8)
m.addLayer(ganancia, {'palette': ['#1565C0']}, 'Ganancia manglar', shown=False, opacity=0.8)

# Guardar tabla
import pandas as pd
cambio_df = pd.DataFrame({
    'Categoría': ['Pérdida', 'Estable', 'Ganancia', 'Cambio neto'],
    'Área (km²)': [round(per_km2, 1), round(est_km2, 1), round(gan_km2, 1), round(neto_km2, 1)]
})
cambio_df.to_csv('../outputs/tables/ganancia_perdida_manglar.csv', index=False)

# Re-exportar dashboard con capas nuevas
m.to_html('../outputs/maps/dashboard_CGSM_final.html', title='Dashboard CGSM - Monitoreo de Manglar 2013-2025')
print(f"\nDashboard actualizado con capas de ganancia/pérdida")
print(f"Datos guardados en outputs/tables/ganancia_perdida_manglar.csv")

=== GANANCIA/PÉRDIDA DE MANGLAR ===

  Pérdida:   183.2 km²
  Estable:   690.9 km²
  Ganancia:  259.2 km²
  Cambio neto: +76.0 km²

Dashboard actualizado con capas de ganancia/pérdida
Datos guardados en outputs/tables/ganancia_perdida_manglar.csv


In [12]:
import ee
import geemap
import os

ee.Initialize()

aoi_coords = [
    [-74.8700, 11.0600], [-74.8200, 11.0700], [-74.7500, 11.0650],
    [-74.6800, 11.0600], [-74.6000, 11.0500], [-74.5300, 11.0400],
    [-74.4800, 11.0350], [-74.4300, 11.0300], [-74.3800, 11.0200],
    [-74.3200, 11.0100], [-74.2500, 10.9900], [-74.2000, 10.9500],
    [-74.1700, 10.9000], [-74.1500, 10.8500], [-74.1300, 10.8000],
    [-74.1200, 10.7500], [-74.1100, 10.7000], [-74.1000, 10.6500],
    [-74.1200, 10.6000], [-74.1500, 10.5500], [-74.2000, 10.5000],
    [-74.2500, 10.4500], [-74.3000, 10.4000], [-74.3500, 10.3500],
    [-74.4500, 10.3400], [-74.5500, 10.3800], [-74.6200, 10.4200],
    [-74.7000, 10.4800], [-74.7500, 10.5500], [-74.8000, 10.6500],
    [-74.8500, 10.7500], [-74.8700, 10.8500], [-74.8800, 10.9500],
    [-74.8700, 11.0600]
]
aoi = ee.Geometry.Polygon(aoi_coords)

def mask_s2(image):
    qa = image.select('QA60')
    return image.updateMask(qa.bitwiseAnd(1<<10).eq(0).And(qa.bitwiseAnd(1<<11).eq(0)))

def add_idx(image):
    ndvi = image.normalizedDifference(['B8','B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3','B8']).rename('NDWI')
    return image.addBands([ndvi, ndwi, ndvi.subtract(ndwi).rename('CMRI')])

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',20))
      .map(mask_s2).map(add_idx))

srtm = ee.Image('USGS/SRTMGL1_003').clip(aoi)
elev_mask = srtm.lt(10)
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').clip(aoi)
near_water = jrc.gt(30).fastDistanceTransform().sqrt().multiply(30).lt(3000)

vis_ndvi = {'min':-0.2,'max':0.8,'palette':['#8B0000','#D32F2F','#FF6F00','#FDD835','#7CB342','#2E7D32','#1B5E20']}
vis_change = {'min':-0.4,'max':0.4,'palette':['#d73027','#f46d43','#fdae61','#ffffbf','#a6d96a','#66bd63','#1a9850']}

ndvi_deg = s2.filterDate('2020-07-01','2020-12-31').select('NDVI').median().clip(aoi)
ndvi_rec = s2.filterDate('2022-01-01','2022-06-30').select('NDVI').median().clip(aoi)
ndvi_act = s2.filterDate('2024-07-01','2025-06-30').select('NDVI').median().clip(aoi)
ndvi_change = ndvi_act.subtract(ndvi_deg)

def mang(s,e):
    return s2.filterDate(s,e).median().clip(aoi).normalizedDifference(['B8','B4']).gt(0.70).And(elev_mask).And(near_water).selfMask()

md = mang('2020-07-01','2020-12-31')
mr = mang('2022-01-01','2022-06-30')
ma = mang('2024-07-01','2025-06-30')

db = md.unmask(0).gt(0)
ab = ma.unmask(0).gt(0)
perdida = db.And(ab.Not()).selfMask()
estable = db.And(ab).selfMask()
ganancia = db.Not().And(ab).selfMask()

s1d = ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi).filterDate('2020-01-01','2020-03-31').filter(ee.Filter.eq('instrumentMode','IW')).select('VH').median().clip(aoi)
s1f = ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi).filterDate('2020-09-01','2020-10-31').filter(ee.Filter.eq('instrumentMode','IW')).select('VH').median().clip(aoi)
sar_diff = s1d.subtract(s1f)

styled_aoi = ee.FeatureCollection([ee.Feature(aoi)]).style(color='FF3333',fillColor='00000000',width=2)

stations = {'Isla Boquerón':(-74.298,10.962,'I'),'Punta Cerro':(-74.283,10.973,'I'),'Punta Chino':(-74.305,10.912,'I'),'Río Sevilla':(-74.325,10.880,'I'),'Caño Palos':(-74.471,10.758,'I'),'CP Pajarales':(-74.75,10.85,'C'),'Caño Clarín':(-74.55,10.55,'C'),'VIPIS':(-74.65,11.02,'C')}
ip = [ee.Feature(ee.Geometry.Point([lon,lat]).buffer(500)) for n,(lon,lat,t) in stations.items() if t=='I']
cp = [ee.Feature(ee.Geometry.Point([lon,lat]).buffer(500)) for n,(lon,lat,t) in stations.items() if t=='C']
ist = ee.FeatureCollection(ip).style(color='E91E63',fillColor='E91E6399',width=2)
cst = ee.FeatureCollection(cp).style(color='FF9800',fillColor='FF980099',width=2)

# --- MAPA con basemap topográfico ---
m = geemap.Map(center=[10.72,-74.45], zoom=10, draw_control=False, measure_control=False,
               fullscreen_control=True, scale_control=True, layout={'height':'700px'},
               basemap='Esri.WorldTopoMap')

# NDVI
m.addLayer(ndvi_act, vis_ndvi, 'NDVI Actual (2024-2025)', shown=False)
m.addLayer(ndvi_deg, vis_ndvi, 'NDVI Degradación (2020)', shown=False)
m.addLayer(ndvi_rec, vis_ndvi, 'NDVI Recuperación (2022)', shown=False)
m.addLayer(ndvi_change, vis_change, 'Cambio NDVI', shown=False)

# Manglar por período
m.addLayer(md, {'palette':['#E57373']}, 'Manglar Degradación (2020)', shown=False, opacity=0.75)
m.addLayer(mr, {'palette':['#FFB74D']}, 'Manglar Recuperación (2022)', shown=False, opacity=0.75)
m.addLayer(ma, {'palette':['#81C784']}, 'Manglar Actual (2024-2025)', shown=False, opacity=0.75)

# Ganancia / pérdida
m.addLayer(perdida, {'palette':['#EF5350']}, 'Pérdida manglar', shown=True, opacity=0.8)
m.addLayer(estable, {'palette':['#66BB6A']}, 'Manglar estable', shown=True, opacity=0.8)
m.addLayer(ganancia, {'palette':['#42A5F5']}, 'Ganancia manglar', shown=True, opacity=0.8)

# SAR
m.addLayer(sar_diff.gt(3).selfMask(), {'palette':['#4DD0E1']}, 'Inundación agua abierta', shown=False, opacity=0.7)
m.addLayer(sar_diff.lt(-2).selfMask(), {'palette':['#CE93D8']}, 'Inundación bajo dosel', shown=False, opacity=0.7)

# AOI y estaciones
m.addLayer(styled_aoi, {}, 'Área de estudio')
m.addLayer(ist, {}, 'Estaciones INVEMAR')
m.addLayer(cst, {}, 'Estaciones complementarias')

# Exportar ANTES de mostrar
os.makedirs('../outputs/maps', exist_ok=True)
m.to_html('../outputs/maps/dashboard_CGSM_final.html', title='Dashboard CGSM - Monitoreo de Manglar 2013-2025')
print("Dashboard exportado")
print("Tamaño:", round(os.path.getsize('../outputs/maps/dashboard_CGSM_final.html')/1e6, 1), "MB")
m

Dashboard exportado
Tamaño: 2.7 MB


Map(center=[10.72, -74.45], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…